In this notebook, we build a baseline model to predict the frames win percentage by player1. In particular, the model predict the player with higher elo rating to be the winner. Then it predicts the winner to have the average win percentage p in the training set and 1-p for the loser. We record the mse on the test set.

In [1]:
import pandas as pd
import numpy as np

In [2]:
#Import the match data for the last 20 tournaments
data = pd.read_csv('/Users/tliu/Desktop/Erdos Project/3_Player_Data_Generation/match_data_20_tourns_modified.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1914 entries, 0 to 1913
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player1                   1914 non-null   object 
 1   player2                   1914 non-null   object 
 2   best_of                   1914 non-null   int64  
 3   player1_elo               1914 non-null   int64  
 4   player2_elo               1914 non-null   int64  
 5   elo_match_win_rate        1914 non-null   float64
 6   elo_frame_win_rate        1914 non-null   float64
 7   p1_matches_played         1914 non-null   int64  
 8   p1_matches_won            1914 non-null   int64  
 9   p1_frames_played          1914 non-null   int64  
 10  p1_frames_won             1914 non-null   int64  
 11  p2_matches_played         1914 non-null   int64  
 12  p2_matches_won            1914 non-null   int64  
 13  p2_frames_played          1914 non-null   int64  
 14  p2_frame

In [3]:
data.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p1_frames_played_3_years,p1_frames_won_3_years,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage
0,Long Zehuang,Haydon Pinhey,7,1328,1158,0.719210,0.604679,78,39,443,...,284,147,121,57,347,168,4,3,0.0,0.571429
1,Wang Yuchen,Andrew Pagett,7,1162,1162,0.500000,0.500000,100,37,626,...,158,89,144,65,417,170,4,1,0.0,0.800000
2,Ben Mertens,Daniel Womersley,7,1262,1109,0.699444,0.594476,107,50,642,...,471,240,86,40,180,89,1,4,1.0,0.200000
3,Paul Deaville,Jimmy White,7,1096,1141,0.438735,0.471905,32,17,158,...,126,66,85,31,384,153,3,4,1.0,0.428571
4,Alexander Ursenbacher,Mostafa Dorgham,7,1302,1031,0.821305,0.663180,286,129,1728,...,400,196,108,41,154,53,4,1,0.0,0.800000


In [4]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2,
                                        shuffle = True,
                                        random_state=216)

In [5]:
# find the average win percentage of winner from training set.
winner_win_percs = []
for i in range(len(data_train)):
    match = data_train.iloc[i]
    match_result = match['match_result']
    win_perc = match['win_percentage']
    if match_result:
        winner_win_percs.append(1-win_perc)
    else: 
        winner_win_percs.append(win_perc)

print(winner_win_percs[:5])


[np.float64(0.5555555555555556), np.float64(0.7142857142857143), np.float64(0.6666666666666666), np.float64(0.6666666666666666), np.float64(0.6666666666666667)]


In [6]:
average_winner_win_perc = np.mean(winner_win_percs)
average_winner_win_perc


np.float64(0.7498339304450492)

In [7]:
#Making prediction on test set.
win_per_prediction = np.zeros(len(data_test))
for i in range(len(data_test)):
    match = data_test.iloc[i]
    if match['elo_match_win_rate']>=0.5:
        win_per_prediction[i] = average_winner_win_perc
    else: 
        win_per_prediction[i] = 1 - average_winner_win_perc

print(win_per_prediction[:5])


[0.74983393 0.74983393 0.25016607 0.74983393 0.74983393]


In [ ]:
#Calculate rmse
from sklearn.metrics import root_mean_squared_error
score = root_mean_squared_error(win_per_prediction, data_test['win_percentage'].values)
print(score)

0.3339174075052421
